In [1]:
import pandas as pd
import numpy as np
import itertools
from pymatgen.core import Element

# 1. Define your search space
# Add as many elements as you want here!
a_elements = ['Cs', 'Rb', 'K', 'Na', 'Li', 'Ba', 'Sr', 'Ca']
b_elements = ['Pb', 'Sn', 'Ge', 'Ti', 'Zn', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zr', 'Bi']
x_elements = ['I', 'Br', 'Cl', 'F', 'O', 'S']
organic_cations = ['MA', 'FA'] # Molecules Pymatgen doesn't have

# 2. Special data for Organic Molecules (Required for Hybrid Perovskites)
organic_data = {
    'MA': {'r': 2.17, 'chi': 2.55, 'EA': 0.60, 'mass': 32.06},
    'FA': {'r': 2.53, 'chi': 2.57, 'EA': 0.50, 'mass': 45.06}
}

def get_atomic_data(symbol):
    """Automatically fetches real data using Pymatgen"""
    if symbol in organic_data:
        return organic_data[symbol]
    
    el = Element(symbol)
    return {
        'r': el.average_ionic_radius, # Pymatgen's real average ionic radius
        'chi': el.X,                  # Pauling Electronegativity
        'EA': el.electron_affinity if el.electron_affinity else 0.0,
        'mass': float(el.atomic_mass)
    }

# 3. Combinatorial Generation
a_sites = a_elements + organic_cations
all_combinations = list(itertools.product(a_sites, b_elements, x_elements))

candidates = []
print(f"Generating data for {len(all_combinations)} materials...")

for a, b, x in all_combinations:
    try:
        data_a = get_atomic_data(a)
        data_b = get_atomic_data(b)
        data_x = get_atomic_data(x)
        
        # Core Features
        rA, rB, rX = data_a['r'], data_b['r'], data_x['r']
        
        # Calculate Tolerance and Octahedral Factors
        t = (rA + rX) / (np.sqrt(2) * (rB + rX))
        mu = rB / rX
        
        # Estimate Structural Properties
        a_lat = 2 * (rB + rX) # Standard cubic approximation
        vol = a_lat**3
        # Density = Mass / (Volume * Avogadro)
        total_mass = data_a['mass'] + data_b['mass'] + 3 * data_x['mass']
        density = total_mass / (0.6022 * vol)

        candidates.append({
            'formula': f"{a}{b}{x}3",
            'rA': rA, 'rB': rB, 'rX': rX,
            'chi_A': data_a['chi'], 'chi_B': data_b['chi'], 'chi_X': data_x['chi'],
            'EA_A': data_a['EA'], 'EA_B': data_b['EA'], 'EA_X': data_x['EA'],
            'tolerance_factor': t, 'octahedral_factor': mu,
            'a': a_lat, 'b': a_lat, 'c': a_lat,
            'volume': vol, 'density': density
        })
    except Exception as e:
        # Skip elements that might have missing data in Pymatgen (rare)
        continue

# 4. Save to CSV
df_candidates = pd.DataFrame(candidates)
df_candidates.to_csv('Candidate_Materials_Pymatgen.csv', index=False)
print("Done! File 'Candidate_Materials_Pymatgen.csv' created.")

Generating data for 720 materials...
Done! File 'Candidate_Materials_Pymatgen.csv' created.


In [2]:
##large dataset version with more elements and features

import pandas as pd
import numpy as np
import itertools
from pymatgen.core import Element

# 1. Expand your element lists to be "Generic"
# This covers almost all stable metals and halides/chalcogens
a_list = ['Cs', 'Rb', 'K', 'Na', 'Li', 'MA', 'FA', 'Ba', 'Sr', 'Ca', 'Mg']
b_list = [el.symbol for el in Element if el.is_metal or el.is_metalloid]
x_list = ['I', 'Br', 'Cl', 'F', 'O', 'S', 'Se']

# 2. Add Molecular data (MA/FA)
molecular_data = {
    'MA': {'r': 2.17, 'chi': 2.55, 'mass': 32.06},
    'FA': {'r': 2.53, 'chi': 2.57, 'mass': 45.06}
}

candidates = []

# 3. Generate 100k+ combinations
# To get even more, use itertools.product(a_list, b_list, b_list, x_list) for double perovskites
for a, b, x in itertools.product(a_list, b_list, x_list):
    try:
        # Get A-site data
        if a in molecular_data:
            rA, chiA, mA = molecular_data[a]['r'], molecular_data[a]['chi'], molecular_data[a]['mass']
        else:
            el_a = Element(a)
            rA, chiA, mA = el_a.average_ionic_radius, el_a.X, float(el_a.atomic_mass)
        
        # Get B and X data
        el_b, el_x = Element(b), Element(x)
        rB, chiB, mB = el_b.average_ionic_radius, el_b.X, float(el_b.atomic_mass)
        rX, chiX, mX = el_x.average_ionic_radius, el_x.X, float(el_x.atomic_mass)

        # Skip if Pymatgen radius is missing
        if not all([rA, rB, rX]): continue

        # 4. Feature Engineering (Same as paper [cite: 226, 225])
        t = (rA + rX) / (1.414 * (rB + rX))
        mu = rB / rX
        a_lat = 2 * (rB + rX)
        density = (mA + mB + 3*mX) / (0.6022 * (a_lat**3))

        candidates.append({
            'formula': f"{a}{b}{x}3",
            'rA': rA, 'rB': rB, 'rX': rX,
            'chi_A': chiA, 'chi_B': chiB, 'chi_X': chiX,
            'tolerance_factor': t, 'octahedral_factor': mu,
            'volume': a_lat**3, 'density': density
        })
    except:
        continue

df_gen = pd.DataFrame(candidates)
df_gen.to_csv('Generic_Candidate_Screening.csv', index=False)
print(f"Generated {len(df_gen)} generic candidates for screening.")

C:\Users\HP\AppData\Local\Temp\ipykernel_20836\3388199543.py:35: UserWarning: No Pauling electronegativity for Rf. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  rB, chiB, mB = el_b.average_ionic_radius, el_b.X, float(el_b.atomic_mass)
C:\Users\HP\AppData\Local\Temp\ipykernel_20836\3388199543.py:35: UserWarning: No Pauling electronegativity for Db. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  rB, chiB, mB = el_b.average_ionic_radius, el_b.X, float(el_b.atomic_mass)
C:\Users\HP\AppData\Local\Temp\ipykernel_20836\3388199543.py:35: UserWarning: No Pauling electronegativity for Sg. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  rB, chiB, mB = el_b.average_ionic_radius, el_b.X, float(el_b.atomic_mass)
C:\Users\HP\AppData\Local\Temp\ipykernel_20836\3388199543.py:35: UserW

Generated 6160 generic candidates for screening.


In [3]:
import pandas as pd
import numpy as np
import itertools
from pymatgen.core import Element

# 1. Expand the element lists (Aiming for high diversity like the 68 elements in the paper)
# A-sites: Alkali, Alkaline Earths, Lanthanides, and Organic Cations
a_list = ['Cs', 'Rb', 'K', 'Na', 'Li', 'Ba', 'Sr', 'Ca', 'Mg', 'La', 'Nd', 'Sm', 'Gd', 'MA', 'FA']
# B-sites: Transition metals and post-transition metals
b_list = ['Pb', 'Sn', 'Ge', 'Ti', 'Zn', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zr', 'Bi', 'In', 'Ga', 'Al', 'V', 'Cr', 'Mo', 'W', 'Sc', 'Y', 'Nb', 'Ta', 'Hf']
# X-sites: Halides and Chalcogens (The paper used Oxides, but for solar we add Halides)
x_list = ['I', 'Br', 'Cl', 'O']

molecular_data = {
    'MA': {'r': 2.17, 'chi': 2.55, 'mass': 32.06},
    'FA': {'r': 2.53, 'chi': 2.57, 'mass': 45.06}
}

def get_data(symbol):
    if symbol in molecular_data: return molecular_data[symbol]
    el = Element(symbol)
    return {'r': el.average_ionic_radius, 'chi': el.X, 'mass': float(el.atomic_mass)}

candidates = []

# 2. GENERATING DOUBLE PEROVSKITES (A2 B B' X6)
# This loop alone will generate tens of thousands of rows
print("Generating Double Perovskite candidates...")
for a, b1, b2, x in itertools.product(a_list, b_list, b_list, x_list):
    try:
        # Get physical data
        d_a, d_b1, d_b2, d_x = get_data(a), get_data(b1), get_data(b2), get_data(x)
        
        # Following the paper's method for "Symmetric Compound Features" [cite: 210]
        # Average Radius for the B-site: (rB + rB') / 2
        rB_avg = (d_b1['r'] + d_b2['r']) / 2
        chiB_avg = (d_b1['chi'] + d_b2['chi']) / 2
        
        # Basic Features
        rA, rX = d_a['r'], d_x['r']
        t = (rA + rX) / (1.414 * (rB_avg + rX))
        mu = rB_avg / rX
        
        # Structure Estimation
        a_lat = 2 * (rB_avg + rX)
        vol = a_lat**3
        total_mass = (2*d_a['mass'] + d_b1['mass'] + d_b2['mass'] + 6*d_x['mass']) / 2 # Mass per formula unit ABX3
        
        candidates.append({
            'formula': f"{a}2{b1}{b2}{x}6",
            'rA': rA, 'rB': rB_avg, 'rX': rX,
            'chi_A': d_a['chi'], 'chi_B': chiB_avg, 'chi_X': d_x['chi'],
            'tolerance_factor': t, 'octahedral_factor': mu,
            'volume': vol, 'density': total_mass / (0.6022 * vol)
        })
    except: continue

df_final = pd.DataFrame(candidates)
df_final.to_csv('Massive_Candidate_Screening.csv', index=False)
print(f"Success! Generated {len(df_final)} candidates.")

Generating Double Perovskite candidates...
Success! Generated 34560 candidates.
